# Introduction to Agents with LangChain

In this lesson, we'll use **LangChain 0.2** to understand how agents work and how to build a simple tool-using agent.

## Learning goals

- Explain what makes an agent different from a standard LLM call.
- Define tools an agent can use.
- Create and run an agent.
- Understand conversation memory and the agent execution trace.

An agent can choose an action based on a user's request, use a tool, inspect the result, and then decide what to do next.

## 1. From an LLM call to an agent

A standard LLM call generates a response from the input you provide. An **agent** can also decide whether to use a tool and which tool to use.

A typical agent interaction looks like this:

1. **Receive** the user's request.
2. **Decide** whether a tool is needed.
3. **Call** a suitable tool and receive its result.
4. **Respond** to the user, potentially after additional tool calls.

The model chooses the next step; the tools perform specific tasks. An agent is not guaranteed to choose the right tool every time, so inspect its behavior and validate important results.

## 2. Install and initialize

Install the packages in your notebook environment. A virtual environment is recommended to keep dependencies isolated.

In [ ]:
%pip install "langchain<0.3" "langchain-core<0.3" "langchain-community<0.3" "langchain-openai<0.2"

Set your OpenAI API key in the environment before running the next cell. Avoid placing secrets directly in notebooks that may be shared.

In [ ]:
import os
from langchain.chat_models import ChatOpenAI
from langchain.agents import Tool, initialize_agent

# Set OPENAI_API_KEY in your environment before running this cell.
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

## 3. Tools: giving the agent capabilities

A **tool** is a Python function exposed to the model through a name and a description. The function does the work; the model decides when it may be useful.

Tool descriptions matter: they help the agent select the right tool for a task.

In [ ]:
def lookup_shipping_days(country: str) -> str:
    """Returns an estimated shipping time for a destination country."""
    shipping = {
        "Spain": "2–3 business days",
        "France": "3–4 business days",
        "Japan": "5–8 business days",
    }
    return shipping.get(country.strip().title(), "Shipping estimate unavailable.")

tools = [
    Tool(
        name="ShippingLookup",
        func=lookup_shipping_days,
        description="Look up the estimated shipping time for a destination country.",
    )
]

## 4. Create and run an agent

`initialize_agent()` connects the language model with the tools and selects an agent strategy. The conversational ReAct strategy can reason through a request, take actions, and use the tool results when forming its response.

`verbose=True` prints a trace of the agent's actions, which is useful for learning and debugging.

In [ ]:
agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent="chat-conversational-react-description",
    verbose=True,
)

result = agent.run("How long might shipping take to Japan?")
print(result)

### What to notice

- Did the agent select `ShippingLookup`?
- What input did it send to the tool?
- How did the tool's returned value affect the final answer?

The agent should use the tool when it needs information that is available from that function. For a general question it may answer without calling a tool.

## 5. One request can use more than one tool

An agent can call multiple tools when a request requires different kinds of information. Each tool should have a focused purpose.

For example, one tool might look up shipping information while another checks whether an item is in stock. The agent can combine the returned results into one response.

**Key idea:** tools provide capabilities; the agent coordinates their use.

## 6. Conversation memory

An agent does not automatically retain every detail from earlier calls. Memory can pass selected conversation history into later interactions.

`ConversationBufferWindowMemory(k=2)` keeps a recent window of conversation turns. This can help the agent answer follow-up questions, but it is not a permanent or unlimited memory.

In [ ]:
from langchain.memory import ConversationBufferWindowMemory

memory = ConversationBufferWindowMemory(
    memory_key="chat_history",
    k=2,
    return_messages=True,
)

agent_with_memory = initialize_agent(
    tools=tools,
    llm=llm,
    agent="chat-conversational-react-description",
    memory=memory,
    verbose=True,
)

agent_with_memory.run("I'm planning a delivery to France.")
agent_with_memory.run("Which country did I mention?")

## 7. Errors and iteration limits

Tool calls can fail, for example because of invalid input or an unavailable service. Robust applications should handle these cases deliberately.

- `max_iterations` limits how many agent steps can occur in one run.
- `handle_parsing_errors=True` helps the agent recover from certain output-parsing errors.
- **Important:** parsing-error handling does not automatically catch every exception raised inside a tool. Use validation and `try`/`except` inside tools when appropriate, and return a clear error message.

For example, a tool that calls an external API should handle timeouts and service errors rather than assuming every request succeeds.

## Summary

- An **agent** decides which steps to take to answer a request.
- A **tool** is a function the agent can choose to call.
- Clear tool names and descriptions help the agent make better choices.
- **Memory** provides conversation history to support follow-up questions.
- **Verbose traces**, iteration limits, and error handling help you inspect and control agent behavior.

You are now ready to practice creating agents, adding tools and memory, and observing how the agent executes a request.